# K-Means Init vs Trained MFA: Assignment and Intrinsic-Dimension Comparison

MFA training in this repo initializes component means from reservoir-KMeans
centroids. This notebook quantifies **what training changes relative to that
initialization** for a single configuration (default: K=1000, layer 5, q=10),
by comparing two hard partitions of the *same* activation token stream:

- **k-means (init)**: each token assigned to the nearest Euclidean centroid
  (`kmeans_centroid_assignments.pt`, produced by
  `dalg-run-metrics assignments --medoids-path ...`)
- **MFA (trained)**: each token assigned to the argmax-responsibility component
  (`mfa_model_assignments.pt`, produced by
  `dalg-run-metrics assignments --data-dir ...`)

Because the MFA components are initialized *from these exact centroids*,
cluster id `k` of the k-means partition corresponds directly to MFA component
`k` — the notebook verifies this (Section 1.1) and then uses the **identity
mapping** for all per-cluster comparisons.

Sections:

1. Setup and artifact validation
2. Assignment agreement (global metrics + per-cluster Jaccard)
3. Centroid movement: how far the trained means moved from their init centroids
4. Intrinsic dimension: distributions and per-cluster change init → trained
5. Massive activations: rogue-dimension profile of centroids and means
6. Spectral profiles: full variance spectra of the two partitions
7. Learned subspaces: what span(W_k) aligns with
8. Compact conclusion

The notebook is safe to run with missing artifacts: dependent sections report
what is missing and skip.

## 1. Setup and Artifact Validation

In [1]:
from __future__ import annotations

import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from scipy.optimize import linear_sum_assignment

try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception as exc:
    px = None
    go = None
    print(f"Plotly unavailable: {exc}")


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "src/dalg").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO = find_repo_root()

# ---------------------------------------------------------------- parameters
LAYER = 5
K = 1000
Q = 10  # MFA rank; only used to locate the MFA run and its intrinsic dims
EPOCHS = 1
MFA_ASSIGN="ar" # ar - nn: nearest neighbor assignment, ar: argmax responsibility
# ------------------------------------------------- paths built from parameters
# MFA_RUN = REPO / f"dalg-cache/pile_gemma2b_activations/layer{LAYER:02d}_{K}_{Q}_component_sharded_mfa"
MFA_RUN = REPO / "dalg-cache/pile_gemma2b_activations/layer05_1000_10_mfa_1epoch_20260703_1538"
_MFA_ASSIGN_FILES = {"nn": "mfa_model_nearest_centroid_assignments.pt", "ar": "mfa_model_assignments.pt"}
MFA_ASSIGN_PATH = MFA_RUN / _MFA_ASSIGN_FILES[MFA_ASSIGN]
MFA_INIT_CENTROIDS = MFA_RUN / "centroids.pt"

CENTROIDS_DIR = REPO / f"dalg-cache/pile_gemma2b_activations/centroids/k{K}_L{LAYER:02d}"
KMEANS_CENTROIDS = CENTROIDS_DIR / "centroids.pt"
KMEANS_ASSIGN = CENTROIDS_DIR / "kmeans_centroid_assignments.pt"

MFA_ID_PATH = REPO / f"output/experiments/{K}_{LAYER:02d}_{Q}/intrinsic_dims.pt"
KMEANS_ID_PATH = REPO / f"output/experiments/centroids_{K}_{LAYER:02d}/intrinsic_dims.pt"

# ------------------------------------------------------------- plot constants
COLOR_KMEANS = "#2a78d6"  # blue  — k-means init partition
COLOR_MFA = "#1baf7a"     # aqua  — trained MFA partition
COLOR_NEUTRAL = "#52514e" # gray  — derived quantities (deltas, joint scatters)
PLOT_TEMPLATE = "plotly_white"

# ------------------------------------------------------------- plot saving
PLOTS_DIR = REPO / "notebooks/plots"
PLOTS_HTML_DIR = PLOTS_DIR / "html"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_HTML_DIR.mkdir(parents=True, exist_ok=True)
PLOT_TAG = f"L{LAYER:02d}_K{K}_q{Q}_ep{EPOCHS}_{MFA_ASSIGN}"

def save_fig(fig, name: str) -> None:
    fig.update_layout(title_font_size=13, title_x=0.5, margin=dict(t=48))
    path = PLOTS_DIR / f"{name}_{PLOT_TAG}.pdf"
    try:
        fig.write_image(str(path), width=1000, height=550)
    except Exception as exc:
        path = PLOTS_HTML_DIR / f"{name}_{PLOT_TAG}.html"
        fig.write_html(str(path), include_plotlyjs="cdn")
        print(f"PDF export failed ({exc}); saved {path.name} instead")

artifacts = pd.DataFrame(
    [
        {"artifact": "kmeans assignments", "path": str(KMEANS_ASSIGN), "exists": KMEANS_ASSIGN.exists()},
        {"artifact": f"mfa assignments ({MFA_ASSIGN})", "path": str(MFA_ASSIGN_PATH), "exists": MFA_ASSIGN_PATH.exists()},
        {"artifact": "kmeans centroids", "path": str(KMEANS_CENTROIDS), "exists": KMEANS_CENTROIDS.exists()},
        {"artifact": "mfa init centroids", "path": str(MFA_INIT_CENTROIDS), "exists": MFA_INIT_CENTROIDS.exists()},
        {"artifact": "kmeans intrinsic dims", "path": str(KMEANS_ID_PATH), "exists": KMEANS_ID_PATH.exists()},
        {"artifact": "mfa intrinsic dims", "path": str(MFA_ID_PATH), "exists": MFA_ID_PATH.exists()},
        {"artifact": "mfa model", "path": str(MFA_RUN / "mfa_model.pt"), "exists": (MFA_RUN / "mfa_model.pt").exists() or (MFA_RUN / "mfa_model_shards.json").exists()},
    ]
)
display(artifacts)

HAVE_ASSIGNMENTS = KMEANS_ASSIGN.exists() and MFA_ASSIGN_PATH.exists()
HAVE_IDS = KMEANS_ID_PATH.exists() and MFA_ID_PATH.exists()
HAVE_MODEL = (
    (MFA_RUN / "mfa_model.pt").exists() or (MFA_RUN / "mfa_model_shards.json").exists()
) and KMEANS_CENTROIDS.exists()

,artifact,path,exists
0,kmeans assignments,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
1,mfa assignments (ar),/orfeo/cephfs/home/dssc/zenocosini/decomposing...,False
2,kmeans centroids,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
3,mfa init centroids,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
4,kmeans intrinsic dims,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
5,mfa intrinsic dims,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
6,mfa model,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True


### 1.1 Cluster Correspondence Check

The per-cluster comparisons below rely on cluster id `k` meaning the same thing
in both partitions. That holds if (and only if) the centroids used for the
nearest-centroid assignments are the same tensors the MFA was initialized from.
We check bit-identity; if it fails, fall back to Hungarian matching before
trusting any per-cluster plot.

In [2]:
IDENTITY_OK = False
if KMEANS_CENTROIDS.exists() and MFA_INIT_CENTROIDS.exists():
    def _centroid_tensor(path: Path) -> torch.Tensor:
        obj = torch.load(path, map_location="cpu")
        if isinstance(obj, dict):
            for key in ("centroids", "mu", "means"):
                if key in obj:
                    obj = obj[key]
                    break
        return obj.float()

    c_km = _centroid_tensor(KMEANS_CENTROIDS)
    c_init = _centroid_tensor(MFA_INIT_CENTROIDS)
    IDENTITY_OK = c_km.shape == c_init.shape and torch.equal(c_km, c_init)
    print(f"kmeans centroids: {tuple(c_km.shape)}, mfa init centroids: {tuple(c_init.shape)}")
    print(f"bit-identical: {IDENTITY_OK}")
    if not IDENTITY_OK:
        print(
            "WARNING: centroids differ -> cluster ids do NOT correspond; "
            "per-cluster sections would need Hungarian matching instead of identity."
        )
    del c_km, c_init
else:
    print("Skipped: missing one or both centroid files; assuming identity correspondence is UNVERIFIED.")

kmeans centroids: (1000, 2048), mfa init centroids: (1000, 2048)
bit-identical: True


## 2. Assignment Agreement

**Experiment.** Both assignment files label every token of the same activation
stream (layer `LAYER`, all shards, prefix-dropped) with one of the same K
cluster ids. The k-means partition is a Voronoi partition around the init
centroids; the MFA partition is the argmax of trained responsibilities, which
moves boundaries according to the learned local subspaces, noise, and mixture
weights. We measure how far training moved tokens across region boundaries:

- **Same-id agreement**: fraction of tokens whose cluster id is unchanged
  (valid because of the identity correspondence verified above).
- **NMI** (normalized mutual information, geometric-mean normalization):
  agreement up to a relabeling — high NMI with lower same-id agreement would
  mean training permuted regions rather than reshaping them.
- **Hungarian sanity check**: the optimal one-to-one matching between the two
  partitions should mostly be the identity if clusters stayed put.
- **Per-cluster Jaccard** (identity mapping): for cluster `k`,
  `J_k = |km_k ∩ mfa_k| / |km_k ∪ mfa_k|` — how much each init region overlaps
  with its trained counterpart.

All statistics derive from the K×K contingency matrix
`C[i, j] = #tokens with kmeans id i and mfa id j`, computed in one
`bincount` pass so the two ~N-length assignment vectors can be freed
immediately.

**Assignment variant.** `MFA_ASSIGN` in Section 1 selects which MFA-side
partition is compared: `"ar"` = argmax responsibility (the full likelihood
rule), `"nn"` = nearest Euclidean centroid *on the trained means*. Comparing
the two runs of this notebook decomposes the disagreement into "the means
moved" (`nn` vs k-means) and "the metric changed" (`ar` vs `nn`).

In [3]:
if not HAVE_ASSIGNMENTS:
    display(Markdown("**Skipped:** missing one or both assignment artifacts."))
else:
    def load_assignments(path: Path) -> dict:
        obj = torch.load(path, map_location="cpu")
        obj["assignments"] = obj["assignments"].to(torch.long)
        obj["cluster_sizes"] = obj["cluster_sizes"].to(torch.long)
        return obj

    km_obj = load_assignments(KMEANS_ASSIGN)
    mfa_obj = load_assignments(MFA_ASSIGN_PATH)
    a_km = km_obj["assignments"]
    a_mfa = mfa_obj["assignments"]
    assert int(km_obj["K"]) == int(mfa_obj["K"]) == K, (km_obj["K"], mfa_obj["K"], K)
    assert a_km.numel() == a_mfa.numel(), (a_km.numel(), a_mfa.numel())
    N = a_km.numel()
    print(f"Loaded {N:,} token assignments for both partitions")

    contingency = (
        torch.bincount(a_km * K + a_mfa, minlength=K * K).reshape(K, K).numpy().astype(np.int64)
    )
    km_sizes = contingency.sum(axis=1)
    mfa_sizes = contingency.sum(axis=0)
    del a_km, a_mfa, km_obj["assignments"], mfa_obj["assignments"]
    gc.collect()

    same_id_agreement = contingency.trace() / N

    def entropy_from_counts(counts: np.ndarray) -> float:
        p = counts[counts > 0].astype(np.float64) / counts.sum()
        return float(-(p * np.log(p)).sum())

    H_km = entropy_from_counts(km_sizes)
    H_mfa = entropy_from_counts(mfa_sizes)
    nz_i, nz_j = np.nonzero(contingency)
    n_ij = contingency[nz_i, nz_j].astype(np.float64)
    p_ij = n_ij / N
    p_i = km_sizes[nz_i].astype(np.float64) / N
    p_j = mfa_sizes[nz_j].astype(np.float64) / N
    MI = float((p_ij * np.log(p_ij / (p_i * p_j))).sum())
    NMI = MI / np.sqrt(H_km * H_mfa)

    row_ind, col_ind = linear_sum_assignment(-contingency)
    hungarian_agreement = contingency[row_ind, col_ind].sum() / N
    frac_self_matched = float((row_ind == col_ind).mean())

    global_metrics = pd.DataFrame(
        [
            {"metric": "tokens (N)", "value": f"{N:,}"},
            {"metric": "same-id agreement", "value": f"{same_id_agreement:.2%}"},
            {"metric": "NMI", "value": f"{NMI:.2%}"},
            {"metric": "Hungarian matched agreement", "value": f"{hungarian_agreement:.2%}"},
            {"metric": "fraction of clusters self-matched (Hungarian)", "value": f"{frac_self_matched:.1%}"},
        ]
    )
    display(global_metrics)

**Skipped:** missing one or both assignment artifacts.

### 2.1 Per-Cluster Jaccard (identity mapping)

One row per cluster id: size in each partition, shared tokens, Jaccard,
recall (fraction of the init region kept by its MFA counterpart), and precision
(fraction of the MFA region coming from its init counterpart).

In [4]:
if not HAVE_ASSIGNMENTS:
    display(Markdown("**Skipped:** missing one or both assignment artifacts."))
else:
    shared = np.diag(contingency)
    per_cluster = pd.DataFrame(
        {
            "cluster": np.arange(K),
            "km_size": km_sizes,
            "mfa_size": mfa_sizes,
            "shared_tokens": shared,
            "jaccard": shared / np.maximum(km_sizes + mfa_sizes - shared, 1),
            "km_recall": shared / np.maximum(km_sizes, 1),
            "mfa_precision": shared / np.maximum(mfa_sizes, 1),
        }
    )
    display(per_cluster[["jaccard", "km_recall", "mfa_precision"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))
    display(per_cluster.sort_values("jaccard").head(10))

    if px is not None:
        fig = px.histogram(
            per_cluster,
            x="jaccard",
            nbins=50,
            title="Per-cluster Jaccard between init (k-means) and trained (MFA) regions",
            color_discrete_sequence=[COLOR_KMEANS],
            template=PLOT_TEMPLATE,
        )
        fig.update_layout(xaxis_title="Jaccard (identity mapping)", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "jaccard_hist")
        fig.show()

        fig = px.scatter(
            per_cluster,
            x="km_size",
            y="jaccard",
            log_x=True,
            hover_data=["cluster", "mfa_size", "shared_tokens"],
            title="Jaccard vs init cluster size",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
            opacity=0.55,
        )
        fig.update_layout(xaxis_title="k-means cluster size (tokens, log)", yaxis_title="Jaccard")
        save_fig(fig, "jaccard_vs_cluster_size")
        fig.show()

**Skipped:** missing one or both assignment artifacts.

## 3. 

Centroid Movement: Init → Trained Means

**Experiment.** Each MFA mean `mu_k` starts at init centroid `c_k` (verified in
Section 1.1) and is free to move during training. We measure how far each one
went:

- **Displacement** `‖mu_k − c_k‖`, shown raw against a baseline of the average
  pairwise distance *between* centroids (within the k-means set and within the
  MFA means) — movement is only "small" if it is small relative to how far
  centroids are from each other; and a scale-free version dividing by the
  distance from `c_k` to its *nearest other* init centroid — a relative
  displacement > 1 means the mean moved farther than the local inter-centroid
  spacing, i.e. well out of its original Voronoi cell.
- **Cosine similarity** between `mu_k` and `c_k`, in two versions. The **raw**
  cosine is inflated: all activations share a massive offset concentrated in a
  couple of rogue dimensions (the global centroid mean has norm ≈ the centroid
  norms themselves), so even unrelated centroids score ≈0.95. The **centered**
  cosine removes the global mean `m` of the init centroids first,
  `cos(mu_k − m, c_k − m)`, and is the informative one: its cross-centroid
  baseline sits near 0. Both plots carry their own baselines (average pairwise
  cosine between *different* centroids within each set).
- **Nearest-init retention**: is `mu_k` still closer to `c_k` than to any other
  init centroid? Together with the Hungarian check of Section 2, this separates
  "means drifted locally" from "means migrated to other regions".
- **Movement vs membership change**: do clusters whose mean moved most also
  have the lowest assignment Jaccard (Section 2.1)?

In [5]:
if not HAVE_MODEL:
    display(Markdown("**Skipped:** missing MFA model files or k-means centroids."))
else:
    from scipy.spatial.distance import cdist

    from dalg.models.mfa import load_mfa

    model = load_mfa(MFA_RUN / "mfa_model.pt", map_location="cpu")
    mu = model.mu.detach().float().cpu().numpy()
    del model
    gc.collect()

    c_init = _centroid_tensor(KMEANS_CENTROIDS).numpy()
    assert mu.shape == c_init.shape, (mu.shape, c_init.shape)

    displacement = np.linalg.norm(mu - c_init, axis=1)

    offdiag = ~np.eye(K, dtype=bool)
    init_pairwise = cdist(c_init, c_init)
    mu_pairwise = cdist(mu, mu)
    avg_dist_km = float(init_pairwise[offdiag].mean())
    avg_dist_mfa = float(mu_pairwise[offdiag].mean())

    np.fill_diagonal(init_pairwise, np.inf)
    nn_init_dist = init_pairwise.min(axis=1)
    rel_displacement = displacement / nn_init_dist

    def _unit(x: np.ndarray) -> np.ndarray:
        return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

    c_unit = _unit(c_init)
    mu_unit = _unit(mu)
    cos_sim = (mu_unit * c_unit).sum(axis=1)
    avg_cos_km = float((c_unit @ c_unit.T)[offdiag].mean())
    avg_cos_mfa = float((mu_unit @ mu_unit.T)[offdiag].mean())

    # raw activations share a massive offset (dominated by a couple of rogue
    # dimensions), which pushes ALL raw cosines toward 1; center by the global
    # mean of the init centroids to expose the informative directions
    global_mean = c_init.mean(axis=0)
    c_cent_unit = _unit(c_init - global_mean)
    mu_cent_unit = _unit(mu - global_mean)
    cos_sim_centered = (mu_cent_unit * c_cent_unit).sum(axis=1)
    avg_cos_km_centered = float((c_cent_unit @ c_cent_unit.T)[offdiag].mean())
    avg_cos_mfa_centered = float((mu_cent_unit @ mu_cent_unit.T)[offdiag].mean())

    nearest_init = cdist(mu, c_init).argmin(axis=1)
    frac_nearest_own_init = float((nearest_init == np.arange(K)).mean())

    move_df = pd.DataFrame(
        {
            "cluster": np.arange(K),
            "displacement": displacement,
            "nn_init_dist": nn_init_dist,
            "rel_displacement": rel_displacement,
            "cos_sim": cos_sim,
            "cos_sim_centered": cos_sim_centered,
            "nearest_own_init": nearest_init == np.arange(K),
        }
    )
    display(move_df[["displacement", "rel_displacement", "cos_sim", "cos_sim_centered"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))
    print(f"avg pairwise distance between k-means centroids: {avg_dist_km:.1f}")
    print(f"avg pairwise distance between MFA means:         {avg_dist_mfa:.1f}")
    print(f"avg pairwise cosine between k-means centroids:   raw {avg_cos_km:.4f} | centered {avg_cos_km_centered:.4f}")
    print(f"avg pairwise cosine between MFA means:           raw {avg_cos_mfa:.4f} | centered {avg_cos_mfa_centered:.4f}")
    print(f"fraction of trained means still nearest their own init centroid: {frac_nearest_own_init:.3f}")

    if px is not None:
        fig = px.histogram(
            move_df,
            x="displacement",
            nbins=60,
            title="Centroid movement ‖mu_k − c_k‖ vs average inter-centroid distance",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
        )
        fig.add_vline(
            x=avg_dist_km, line_dash="dash", line_color=COLOR_KMEANS,
            annotation_text=f"avg dist between k-means centroids ({avg_dist_km:.1f})",
            annotation_position="top left", annotation_font_color=COLOR_KMEANS,
        )
        fig.add_vline(
            x=avg_dist_mfa, line_dash="dot", line_color=COLOR_MFA,
            annotation_text=f"avg dist between MFA means ({avg_dist_mfa:.1f})",
            annotation_position="bottom right", annotation_font_color=COLOR_MFA,
        )
        fig.update_layout(xaxis_title="displacement ‖mu_k − c_k‖", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "displacement_hist")
        fig.show()

        fig = px.histogram(
            move_df,
            x="rel_displacement",
            nbins=60,
            title="Centroid movement relative to local init spacing (‖mu_k − c_k‖ / nearest-init distance)",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
        )
        fig.add_vline(x=1.0, line_dash="dash", line_color="#0b0b0b", annotation_text="moved past nearest init centroid")
        fig.update_layout(xaxis_title="relative displacement", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "rel_displacement_hist")
        fig.show()

        raw_start = float(np.floor(move_df["cos_sim"].min() * 100) / 100)
        fig = px.histogram(
            move_df,
            x="cos_sim",
            title="RAW cosine similarity between trained mean and init centroid (inflated by shared offset)",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
        )
        fig.add_vline(
            x=avg_cos_km, line_dash="dash", line_color=COLOR_KMEANS,
            annotation_text=f"avg cos between k-means centroids ({avg_cos_km:.3f})",
            annotation_position="top left", annotation_font_color=COLOR_KMEANS,
        )
        fig.add_vline(
            x=avg_cos_mfa, line_dash="dot", line_color=COLOR_MFA,
            annotation_text=f"avg cos between MFA means ({avg_cos_mfa:.3f})",
            annotation_position="bottom left", annotation_font_color=COLOR_MFA,
        )
        fig.update_traces(xbins=dict(start=raw_start, end=1.0, size=(1.0 - raw_start) / 60))
        fig.update_layout(xaxis_title="cos(mu_k, c_k)", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "cos_raw_hist")
        fig.show()

        fig = px.histogram(
            move_df,
            x="cos_sim_centered",
            title="CENTERED cosine similarity between trained mean and init centroid (global mean removed)",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
        )
        fig.add_vline(
            x=avg_cos_km_centered, line_dash="dash", line_color=COLOR_KMEANS,
            annotation_text=f"avg centered cos between k-means centroids ({avg_cos_km_centered:.3f})",
            annotation_position="top left", annotation_font_color=COLOR_KMEANS,
        )
        fig.add_vline(
            x=avg_cos_mfa_centered, line_dash="dot", line_color=COLOR_MFA,
            annotation_text=f"avg centered cos between MFA means ({avg_cos_mfa_centered:.3f})",
            annotation_position="bottom left", annotation_font_color=COLOR_MFA,
        )
        fig.update_traces(xbins=dict(start=-1.0, end=1.0, size=0.025))
        fig.update_layout(xaxis_title="cos(mu_k − m, c_k − m)", yaxis_title="clusters", bargap=0.05, xaxis_range=[-1, 1])
        save_fig(fig, "cos_centered_hist")
        fig.show()

    if HAVE_ASSIGNMENTS and px is not None:
        move_jac = move_df.merge(per_cluster[["cluster", "jaccard", "km_size"]], on="cluster")
        rho = float(move_jac["rel_displacement"].corr(move_jac["jaccard"], method="spearman"))
        print(f"Spearman(rel_displacement, jaccard) = {rho:.3f}")
        fig = px.scatter(
            move_jac,
            x="rel_displacement",
            y="jaccard",
            hover_data=["cluster", "displacement", "cos_sim_centered", "km_size"],
            title="Assignment Jaccard vs centroid movement",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
            opacity=0.55,
        )
        fig.update_layout(xaxis_title="relative displacement of mean", yaxis_title="Jaccard (init vs trained region)")
        save_fig(fig, "jaccard_vs_movement")
        fig.show()

,displacement,rel_displacement,cos_sim,cos_sim_centered
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,12.554927,0.659084,0.995447,0.903707
std,5.047400,0.379115,0.003240,0.104957
min,1.299513,0.060879,0.983106,0.250538
10%,5.891125,0.211712,0.990999,0.760686
25%,8.668921,0.360707,0.993838,0.858843
50%,12.363172,0.586488,0.996145,0.940551
75%,16.296302,0.924653,0.997860,0.980534
90%,19.548634,1.205244,0.999021,0.992625
max,26.584724,1.909082,0.999928,0.999630


avg pairwise distance between k-means centroids: 49.0
avg pairwise distance between MFA means:         53.3
avg pairwise cosine between k-means centroids:   raw 0.9471 | centered 0.0060
avg pairwise cosine between MFA means:           raw 0.9350 | centered 0.0028
fraction of trained means still nearest their own init centroid: 0.965


## 4. Intrinsic Dimension: Init Clusters vs MFA Clusters

**Experiment.** For each partition, `dalg-run-metrics intrinsic-dim` sampled up
to `max_samples` activations per cluster *according to that partition's own
assignments*, ran PCA on the centered samples, and recorded the number of
principal directions needed to reach the variance threshold (typically 90%) of
within-cluster variance. So each file characterizes the local geometry of its
own partition:

- **k-means IDs** describe the Voronoi regions around the init centroids;
- **MFA IDs** describe the regions after training reshaped the boundaries.

We compare (a) the two per-cluster ID **distributions**, and (b) the
**per-cluster change** `ΔID_k = ID_mfa[k] − ID_km[k]` under the identity
correspondence — including whether clusters whose *membership* changed most
(low Jaccard, Section 2) are also the ones whose *dimensionality* changed most.

Clusters with `intrinsic_dim == 0` were skipped by the metric (population below
`min_population`) and are masked out of per-cluster comparisons.

In [6]:
if not HAVE_IDS:
    display(Markdown("**Skipped:** missing one or both intrinsic-dim artifacts."))
else:
    id_km_obj = torch.load(KMEANS_ID_PATH, map_location="cpu")
    id_mfa_obj = torch.load(MFA_ID_PATH, map_location="cpu")

    assert int(id_km_obj["K"]) == int(id_mfa_obj["K"]) == K, (id_km_obj["K"], id_mfa_obj["K"], K)
    assert id_km_obj["variance_threshold"] == id_mfa_obj["variance_threshold"], (
        id_km_obj["variance_threshold"],
        id_mfa_obj["variance_threshold"],
    )
    VAR_THRESHOLD = float(id_km_obj["variance_threshold"])

    id_km = id_km_obj["intrinsic_dims"].to(torch.long).numpy()
    id_mfa = id_mfa_obj["intrinsic_dims"].to(torch.long).numpy()
    valid = (id_km > 0) & (id_mfa > 0)
    print(
        f"variance threshold: {VAR_THRESHOLD}, "
        f"model_kind: kmeans={id_km_obj.get('model_kind')}, mfa={id_mfa_obj.get('model_kind')}"
    )
    print(f"clusters with valid ID in both: {int(valid.sum())}/{K}")

    id_summary = pd.DataFrame(
        [
            {
                "partition": name,
                "mean": vals.mean(),
                "median": np.median(vals),
                "p10": np.quantile(vals, 0.1),
                "p90": np.quantile(vals, 0.9),
                "min": vals.min(),
                "max": vals.max(),
            }
            for name, vals in [("kmeans (init)", id_km[valid]), ("mfa (trained)", id_mfa[valid])]
        ]
    )
    display(id_summary)

    if px is not None:
        id_long = pd.DataFrame(
            {
                "intrinsic_dim": np.concatenate([id_km[valid], id_mfa[valid]]),
                "partition": ["kmeans (init)"] * int(valid.sum()) + ["mfa (trained)"] * int(valid.sum()),
            }
        )
        fig = px.histogram(
            id_long,
            x="intrinsic_dim",
            color="partition",
            nbins=60,
            barmode="overlay",
            opacity=0.6,
            title=f"Per-cluster intrinsic dimension ({VAR_THRESHOLD:.0%} variance threshold)",
            color_discrete_map={"kmeans (init)": COLOR_KMEANS, "mfa (trained)": COLOR_MFA},
            template=PLOT_TEMPLATE,
        )
        fig.update_layout(xaxis_title="intrinsic dimension", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "intrinsic_dim_hist")
        fig.show()

variance threshold: 0.9, model_kind: kmeans=assignments, mfa=mfa
clusters with valid ID in both: 1000/1000


,partition,mean,median,p10,p90,min,max
0,kmeans (init),246.231,237.0,124.9,382.1,1,582
1,mfa (trained),202.478,205.0,106.0,298.0,2,498


KeyboardInterrupt: 

### 4.1 Per-Cluster Change: Init → Trained

Same cluster id on both axes. Points below the diagonal are clusters whose
intrinsic dimension *dropped* during training.

In [ ]:
if not HAVE_IDS:
    display(Markdown("**Skipped:** missing one or both intrinsic-dim artifacts."))
else:
    id_df = pd.DataFrame(
        {
            "cluster": np.arange(K)[valid],
            "id_kmeans": id_km[valid],
            "id_mfa": id_mfa[valid],
            "km_cluster_size": id_km_obj["cluster_sizes"].numpy()[valid],
        }
    )
    id_df["delta_id"] = id_df["id_mfa"] - id_df["id_kmeans"]
    display(id_df["delta_id"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

    if px is not None:
        lim = float(max(id_df["id_kmeans"].max(), id_df["id_mfa"].max())) * 1.05
        fig = px.scatter(
            id_df,
            x="id_kmeans",
            y="id_mfa",
            hover_data=["cluster", "delta_id", "km_cluster_size"],
            title="Per-cluster intrinsic dimension: init vs trained",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
            opacity=0.55,
        )
        fig.add_shape(type="line", x0=0, y0=0, x1=lim, y1=lim, line=dict(color="#0b0b0b", dash="dash", width=1))
        fig.update_layout(
            xaxis_title="intrinsic dim (k-means init partition)",
            yaxis_title="intrinsic dim (trained MFA partition)",
            xaxis_range=[0, lim],
            yaxis_range=[0, lim],
        )
        save_fig(fig, "id_init_vs_trained")
        fig.show()

        fig = px.histogram(
            id_df,
            x="delta_id",
            nbins=60,
            title="ΔID = ID(mfa) − ID(kmeans) per cluster",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
        )
        fig.add_vline(x=0, line_dash="dash", line_color="#0b0b0b")
        fig.update_layout(xaxis_title="ΔID (trained − init)", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "delta_id_hist")
        fig.show()

        fig = px.scatter(
            id_df,
            x="km_cluster_size",
            y="delta_id",
            log_x=True,
            hover_data=["cluster", "id_kmeans", "id_mfa"],
            title="ΔID vs init cluster size",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
            opacity=0.55,
        )
        fig.add_hline(y=0, line_dash="dash", line_color="#0b0b0b")
        fig.update_layout(xaxis_title="k-means cluster size (tokens, log)", yaxis_title="ΔID")
        save_fig(fig, "delta_id_vs_cluster_size")
        fig.show()

count    1000.000000
mean      -43.753000
std        74.387684
min      -460.000000
10%      -140.000000
25%       -75.000000
50%       -24.000000
75%         1.000000
90%        25.000000
max       309.000000
Name: delta_id, dtype: float64

### 4.2 Does Membership Change Predict Dimensionality Change?

Clusters whose token membership training reshaped the most (low Jaccard) might
also be the ones whose measured intrinsic dimension changed most. Requires both
the assignment (Section 2) and intrinsic-dim (Section 4) artifacts.

In [ ]:
if not (HAVE_IDS and HAVE_ASSIGNMENTS):
    display(Markdown("**Skipped:** requires both assignment and intrinsic-dim artifacts."))
else:
    merged = id_df.merge(per_cluster[["cluster", "jaccard"]], on="cluster")
    merged["abs_delta_id"] = merged["delta_id"].abs()
    corr = merged[["jaccard", "delta_id", "abs_delta_id"]].corr(method="spearman")
    print("Spearman correlations:")
    display(corr)

    if px is not None:
        fig = px.scatter(
            merged,
            x="jaccard",
            y="delta_id",
            hover_data=["cluster", "id_kmeans", "id_mfa", "km_cluster_size"],
            title="ΔID vs assignment Jaccard per cluster",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
            opacity=0.55,
        )
        fig.add_hline(y=0, line_dash="dash", line_color="#0b0b0b")
        fig.update_layout(xaxis_title="Jaccard (init vs trained region)", yaxis_title="ΔID (trained − init)")
        save_fig(fig, "delta_id_vs_jaccard")
        fig.show()

Spearman correlations:


,jaccard,delta_id,abs_delta_id
jaccard,1.000000,0.566446,-0.669335
delta_id,0.566446,1.000000,-0.740981
abs_delta_id,-0.669335,-0.740981,1.000000


## 5. Massive Activations in K-Means Centroids and MFA Means

**Measurement.** A massive activation coordinate is a feature dimension whose RMS across centroids is far above the typical feature RMS. Concretely, for a centroid matrix `X` with shape `(K, D)`, define `dim_rms[d] = sqrt(mean_k X[k, d]^2)` and mark dimension `d` massive when `dim_rms[d] / median(dim_rms) >= MASSIVE_RMS_RATIO`.

This separates the question into two checks:

1. **Do massive coordinates exist?** Count dimensions above the RMS ratio threshold for K-means centroids and MFA means.
2. **Does each centroid have the same massive activations?** For each centroid, build a binary signature over the globally massive dimensions, where a dimension is active for that centroid if `abs(X[k, d]) >= MASSIVE_RMS_RATIO * median(dim_rms)`. If every centroid has the same massive activations, coverage is near 1 and pairwise signature Jaccard is near 1.

The same-index K-means-vs-MFA signature Jaccard then asks whether training preserves the massive-coordinate signature of centroid `k`.

In [ ]:
MASSIVE_RMS_RATIO = 10.0
TOP_MASSIVE_DIMS_TO_SHOW = 30


def _load_centroid_array(path: Path) -> np.ndarray:
    obj = torch.load(path, map_location="cpu")
    if isinstance(obj, dict):
        for key in ("centroids", "mu", "means"):
            if key in obj:
                obj = obj[key]
                break
    return obj.detach().float().cpu().numpy() if isinstance(obj, torch.Tensor) else np.asarray(obj, dtype=np.float32)


def _load_mfa_means(run_dir: Path) -> np.ndarray:
    from dalg.models.mfa import load_mfa

    model = load_mfa(run_dir / "mfa_model.pt", map_location="cpu")
    means = model.mu.detach().float().cpu().numpy()
    del model
    gc.collect()
    return means


def _massive_state(name: str, x: np.ndarray, *, ratio_threshold: float) -> dict:
    x = np.asarray(x, dtype=np.float32)
    dim_rms = np.sqrt(np.mean(np.square(x, dtype=np.float64), axis=0))
    baseline = float(np.median(dim_rms))
    dim_ratio = dim_rms / max(baseline, 1e-12)
    massive_mask = dim_ratio >= ratio_threshold
    massive_dims = np.flatnonzero(massive_mask)
    active_threshold = ratio_threshold * baseline
    active = np.abs(x[:, massive_dims]) >= active_threshold if massive_dims.size else np.zeros((x.shape[0], 0), dtype=bool)
    return {
        "name": name,
        "x": x,
        "dim_rms": dim_rms,
        "baseline": baseline,
        "dim_ratio": dim_ratio,
        "massive_mask": massive_mask,
        "massive_dims": massive_dims,
        "active_threshold": active_threshold,
        "active": active,
    }


def _sample_pairwise_jaccard(active: np.ndarray, *, max_pairs: int = 50000, seed: int = 0) -> dict:
    n, width = active.shape
    if n < 2 or width == 0:
        return {"mean": np.nan, "median": np.nan, "p10": np.nan, "p90": np.nan, "pairs": 0}
    rng = np.random.default_rng(seed)
    total_pairs = n * (n - 1) // 2
    if total_pairs <= max_pairs:
        i, j = np.triu_indices(n, k=1)
    else:
        i = rng.integers(0, n, size=max_pairs)
        j = rng.integers(0, n - 1, size=max_pairs)
        j = j + (j >= i)
    inter = np.logical_and(active[i], active[j]).sum(axis=1)
    union = np.logical_or(active[i], active[j]).sum(axis=1)
    vals = np.divide(inter, union, out=np.full(inter.shape, np.nan, dtype=float), where=union > 0)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return {"mean": np.nan, "median": np.nan, "p10": np.nan, "p90": np.nan, "pairs": int(len(i))}
    return {
        "mean": float(vals.mean()),
        "median": float(np.median(vals)),
        "p10": float(np.quantile(vals, 0.1)),
        "p90": float(np.quantile(vals, 0.9)),
        "pairs": int(len(i)),
    }


def _active_on_dims(state: dict, dims: np.ndarray) -> np.ndarray:
    if dims.size == 0:
        return np.zeros((state["x"].shape[0], 0), dtype=bool)
    return np.abs(state["x"][:, dims]) >= state["active_threshold"]


if not HAVE_MODEL:
    n_massive_km = 0
    n_massive_mfa = 0
    display(Markdown("**Skipped:** missing MFA model files or k-means centroids."))
else:
    if "c_init" not in globals():
        c_init = _load_centroid_array(KMEANS_CENTROIDS)
    if "mu" not in globals():
        mu = _load_mfa_means(MFA_RUN)
    assert c_init.shape == mu.shape == (K, c_init.shape[1]), (c_init.shape, mu.shape, K)

    km_mass = _massive_state("kmeans", c_init, ratio_threshold=MASSIVE_RMS_RATIO)
    mfa_mass = _massive_state("mfa", mu, ratio_threshold=MASSIVE_RMS_RATIO)
    n_massive_km = int(km_mass["massive_mask"].sum())
    n_massive_mfa = int(mfa_mass["massive_mask"].sum())

    shared_massive = km_mass["massive_mask"] & mfa_mass["massive_mask"]
    union_massive = km_mass["massive_mask"] | mfa_mass["massive_mask"]
    massive_dim_jaccard = float(shared_massive.sum() / max(union_massive.sum(), 1))

    dim_df = pd.DataFrame(
        {
            "dim": np.arange(c_init.shape[1]),
            "kmeans_rms": km_mass["dim_rms"],
            "mfa_rms": mfa_mass["dim_rms"],
            "kmeans_rms_ratio": km_mass["dim_ratio"],
            "mfa_rms_ratio": mfa_mass["dim_ratio"],
            "massive_kmeans": km_mass["massive_mask"],
            "massive_mfa": mfa_mass["massive_mask"],
            "massive_shared": shared_massive,
        }
    )

    global_massive_summary = pd.DataFrame(
        [
            {
                "partition": "kmeans",
                "median_dim_rms": km_mass["baseline"],
                "active_value_threshold": km_mass["active_threshold"],
                "massive_dims": n_massive_km,
                "max_rms_ratio": float(km_mass["dim_ratio"].max()),
            },
            {
                "partition": "mfa",
                "median_dim_rms": mfa_mass["baseline"],
                "active_value_threshold": mfa_mass["active_threshold"],
                "massive_dims": n_massive_mfa,
                "max_rms_ratio": float(mfa_mass["dim_ratio"].max()),
            },
        ]
    )
    display(global_massive_summary)
    print(f"shared massive dims: {int(shared_massive.sum())}; union massive dims: {int(union_massive.sum())}; Jaccard: {massive_dim_jaccard:.3f}")

    display(
        dim_df.assign(max_rms_ratio=lambda d: d[["kmeans_rms_ratio", "mfa_rms_ratio"]].max(axis=1))
        .sort_values("max_rms_ratio", ascending=False)
        .head(TOP_MASSIVE_DIMS_TO_SHOW)
    )

    consistency_rows = []
    for state in (km_mass, mfa_mass):
        active = state["active"]
        n_dims = active.shape[1]
        active_count = active.sum(axis=1) if n_dims else np.zeros(state["x"].shape[0], dtype=int)
        coverage = active_count / n_dims if n_dims else np.full(state["x"].shape[0], np.nan)
        pairwise = _sample_pairwise_jaccard(active)
        consistency_rows.append(
            {
                "partition": state["name"],
                "massive_dims": n_dims,
                "median_active_dims_per_centroid": float(np.median(active_count)),
                "min_active_dims_per_centroid": int(active_count.min()) if active_count.size else 0,
                "median_coverage_of_global_massive_dims": float(np.nanmedian(coverage)) if n_dims else np.nan,
                "frac_centroids_with_all_massive_dims": float((active_count == n_dims).mean()) if n_dims else np.nan,
                "mean_pairwise_signature_jaccard": pairwise["mean"],
                "median_pairwise_signature_jaccard": pairwise["median"],
                "sampled_pairs": pairwise["pairs"],
            }
        )
        state["active_count"] = active_count
        state["coverage"] = coverage

    consistency_df = pd.DataFrame(consistency_rows)
    display(consistency_df)

    union_dims = np.flatnonzero(union_massive)
    km_active_union = _active_on_dims(km_mass, union_dims)
    mfa_active_union = _active_on_dims(mfa_mass, union_dims)
    sig_inter = np.logical_and(km_active_union, mfa_active_union).sum(axis=1)
    sig_union = np.logical_or(km_active_union, mfa_active_union).sum(axis=1)
    same_index_signature_jaccard = np.divide(sig_inter, sig_union, out=np.full(K, np.nan, dtype=float), where=sig_union > 0)

    both_active = np.logical_and(km_active_union, mfa_active_union)
    sign_same = np.sign(c_init[:, union_dims]) == np.sign(mu[:, union_dims]) if union_dims.size else np.zeros((K, 0), dtype=bool)
    same_index_sign_agreement = np.divide(
        np.logical_and(both_active, sign_same).sum(axis=1),
        both_active.sum(axis=1),
        out=np.full(K, np.nan, dtype=float),
        where=both_active.sum(axis=1) > 0,
    )

    massive_cluster_df = pd.DataFrame(
        {
            "cluster": np.arange(K),
            "kmeans_active_massive_dims": km_mass["active_count"],
            "mfa_active_massive_dims": mfa_mass["active_count"],
            "kmeans_massive_coverage": km_mass["coverage"],
            "mfa_massive_coverage": mfa_mass["coverage"],
            "same_index_signature_jaccard": same_index_signature_jaccard,
            "same_index_sign_agreement": same_index_sign_agreement,
        }
    )
    display(massive_cluster_df.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

    if px is not None:
        dim_long = dim_df.melt(
            id_vars=["dim"],
            value_vars=["kmeans_rms_ratio", "mfa_rms_ratio"],
            var_name="partition",
            value_name="rms_ratio",
        )
        dim_long["partition"] = dim_long["partition"].map({"kmeans_rms_ratio": "kmeans", "mfa_rms_ratio": "mfa"})
        top_dims = (
            dim_df.assign(max_rms_ratio=lambda d: d[["kmeans_rms_ratio", "mfa_rms_ratio"]].max(axis=1))
            .sort_values("max_rms_ratio", ascending=False)
            .head(TOP_MASSIVE_DIMS_TO_SHOW)["dim"]
        )
        fig = px.bar(
            dim_long[dim_long["dim"].isin(top_dims)],
            x="dim",
            y="rms_ratio",
            color="partition",
            barmode="group",
            title=f"Top coordinate RMS ratios across centroids (threshold = {MASSIVE_RMS_RATIO:g}x median)",
            color_discrete_map={"kmeans": COLOR_KMEANS, "mfa": COLOR_MFA},
            template=PLOT_TEMPLATE,
        )
        fig.add_hline(y=MASSIVE_RMS_RATIO, line_dash="dash", line_color="#0b0b0b")
        fig.update_layout(xaxis_title="activation dimension", yaxis_title="RMS / median dimension RMS")
        save_fig(fig, "massive_dim_rms_ratios")
        fig.show()

        coverage_long = massive_cluster_df.melt(
            id_vars=["cluster"],
            value_vars=["kmeans_massive_coverage", "mfa_massive_coverage"],
            var_name="partition",
            value_name="coverage",
        )
        coverage_long["partition"] = coverage_long["partition"].map(
            {"kmeans_massive_coverage": "kmeans", "mfa_massive_coverage": "mfa"}
        )
        fig = px.histogram(
            coverage_long.dropna(),
            x="coverage",
            color="partition",
            barmode="overlay",
            opacity=0.65,
            nbins=30,
            title="Per-centroid coverage of globally massive dimensions",
            color_discrete_map={"kmeans": COLOR_KMEANS, "mfa": COLOR_MFA},
            template=PLOT_TEMPLATE,
        )
        fig.update_layout(xaxis_title="fraction of global massive dims active in centroid", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "massive_dim_coverage_hist")
        fig.show()

        if union_dims.size:
            fig = px.histogram(
                massive_cluster_df.dropna(subset=["same_index_signature_jaccard"]),
                x="same_index_signature_jaccard",
                nbins=30,
                title="Same-index massive-coordinate signature Jaccard: K-means centroid vs MFA mean",
                color_discrete_sequence=[COLOR_NEUTRAL],
                template=PLOT_TEMPLATE,
            )
            fig.update_layout(xaxis_title="Jaccard over union of massive dims", yaxis_title="clusters", bargap=0.05)
            save_fig(fig, "massive_signature_same_index_jaccard")
            fig.show()


,partition,median_dim_rms,active_value_threshold,massive_dims,max_rms_ratio
0,kmeans,0.207207,2.072068,34,591.945225
1,mfa,0.250555,2.505552,30,487.733459


shared massive dims: 30; union massive dims: 34; Jaccard: 0.882


,dim,kmeans_rms,mfa_rms,kmeans_rms_ratio,mfa_rms_ratio,massive_kmeans,massive_mfa,massive_shared,max_rms_ratio
674,674,122.655063,122.204136,591.945225,487.733459,True,True,True,591.945225
50,50,47.743588,47.833255,230.415185,190.909080,True,True,True,230.415185
1741,1741,7.577142,7.777539,36.568024,31.041223,True,True,True,36.568024
2019,2019,6.238065,6.617169,30.105504,26.410030,True,True,True,30.105504
381,381,5.853880,6.254561,28.251395,24.962810,True,True,True,28.251395
1967,1967,5.455545,5.310925,26.328989,21.196631,True,True,True,26.328989
604,604,5.116141,4.869557,24.690994,19.435070,True,True,True,24.690994
1978,1978,5.041331,5.465189,24.329952,21.812319,True,True,True,24.329952
1870,1870,5.035964,5.448883,24.304051,21.747241,True,True,True,24.304051
876,876,5.009710,5.112720,24.177345,20.405567,True,True,True,24.177345


,partition,massive_dims,median_active_dims_per_centroid,min_active_dims_per_centroid,median_coverage_of_global_massive_dims,frac_centroids_with_all_massive_dims,mean_pairwise_signature_jaccard,median_pairwise_signature_jaccard,sampled_pairs
0,kmeans,34,20.0,12,0.588235,0.0,0.512999,0.500000,50000
1,mfa,30,17.0,10,0.566667,0.0,0.462179,0.454545,50000


,cluster,kmeans_active_massive_dims,mfa_active_massive_dims,kmeans_massive_coverage,mfa_massive_coverage,same_index_signature_jaccard,same_index_sign_agreement
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,499.500000,20.006000,16.628000,0.588412,0.554267,0.788612,0.999756
std,288.819436,2.948671,2.608528,0.086726,0.086951,0.107167,0.003884
min,0.000000,12.000000,10.000000,0.352941,0.333333,0.375000,0.928571
10%,99.900000,16.000000,13.000000,0.470588,0.433333,0.650000,1.000000
25%,249.750000,18.000000,15.000000,0.529412,0.500000,0.720000,1.000000
50%,499.500000,20.000000,17.000000,0.588235,0.566667,0.796552,1.000000
75%,749.250000,22.000000,18.000000,0.647059,0.600000,0.866667,1.000000
90%,899.100000,24.000000,20.000000,0.705882,0.666667,0.916667,1.000000
max,999.000000,28.000000,25.000000,0.823529,0.833333,1.000000,1.000000


## 6. Spectral Profiles of the Two Partitions

**Experiment.** The intrinsic-dim artifacts store the full PCA variance
spectrum of every cluster (`cluster_variances`), so this section costs no new
computation. Section 4 showed the scalar ID (rank to 90% variance) drops from
~381 to ~203; here we look at the *whole spectrum* to see how:

- **Cumulative variance curves**: median and 10–90% band of the normalized
  cumulative spectrum across clusters, per partition. A partition whose curve
  rises faster has regions concentrated in fewer directions everywhere along
  the spectrum, not just at the 90% threshold.
- **Variance fraction in the top q directions** (q = the MFA rank): per-cluster
  distribution. Directly comparable to what a rank-q factor model can capture:
  if MFA regions concentrate much more variance in their top q PCs, the learned
  covariance has an easier job describing its own regions.
- **Spectral participation ratio** `(Σλ)² / Σλ²`: a threshold-free effective
  dimensionality of each cluster's spectrum.


In [ ]:
if not HAVE_IDS:
    display(Markdown("**Skipped:** missing one or both intrinsic-dim artifacts."))
else:
    RANK_GRID = np.arange(1, 1025)

    def spectrum_stats(obj):
        curves, topq, pr = [], [], []
        for k in range(K):
            if not valid[k]:
                continue
            v = obj["cluster_variances"][k].float().numpy()
            if v.size == 0 or v.sum() <= 0:
                continue
            cum = np.cumsum(v) / v.sum()
            curves.append(np.interp(RANK_GRID, np.arange(1, v.size + 1), cum))
            topq.append(cum[min(Q, v.size) - 1])
            pr.append(v.sum() ** 2 / (v ** 2).sum())
        return np.stack(curves), np.asarray(topq), np.asarray(pr)

    curves_km, topq_frac_km, pr_km = spectrum_stats(id_km_obj)
    curves_mfa, topq_frac_mfa, pr_mfa = spectrum_stats(id_mfa_obj)

    def rank_to(curves, thr):
        return (curves < thr).sum(axis=1) + 1

    spectral_summary = pd.DataFrame(
        [
            {
                "partition": name,
                f"median variance frac in top {Q} PCs": float(np.median(tq)),
                "median spectral participation ratio": float(np.median(pr_)),
                "median rank to 50% var": float(np.median(rank_to(cv, 0.5))),
                "median rank to 90% var": float(np.median(rank_to(cv, 0.9))),
            }
            for name, cv, tq, pr_ in [
                ("kmeans (init)", curves_km, topq_frac_km, pr_km),
                ("mfa (trained)", curves_mfa, topq_frac_mfa, pr_mfa),
            ]
        ]
    )
    display(spectral_summary)

    if go is not None:
        fig = go.Figure()
        for name, cv, color, fill in [
            ("kmeans (init)", curves_km, COLOR_KMEANS, "rgba(42,120,214,0.15)"),
            ("mfa (trained)", curves_mfa, COLOR_MFA, "rgba(27,175,122,0.15)"),
        ]:
            med = np.median(cv, axis=0)
            p10 = np.quantile(cv, 0.1, axis=0)
            p90 = np.quantile(cv, 0.9, axis=0)
            fig.add_trace(go.Scatter(x=RANK_GRID, y=p90, line=dict(width=0), showlegend=False, hoverinfo="skip"))
            fig.add_trace(go.Scatter(x=RANK_GRID, y=p10, fill="tonexty", fillcolor=fill, line=dict(width=0), showlegend=False, hoverinfo="skip"))
            fig.add_trace(go.Scatter(x=RANK_GRID, y=med, name=name, line=dict(color=color, width=2)))
        fig.update_layout(
            template=PLOT_TEMPLATE,
            title="Normalized cumulative variance spectrum per cluster (median, 10–90% band)",
            xaxis_title="PCA rank (log)",
            yaxis_title="cumulative variance fraction",
            xaxis_type="log",
        )
        fig.add_hline(y=0.9, line_dash="dash", line_color="#0b0b0b")
        save_fig(fig, "spectral_cumvar_curves")
        fig.show()

    if px is not None:
        topq_long = pd.DataFrame(
            {
                "topq_fraction": np.concatenate([topq_frac_km, topq_frac_mfa]),
                "partition": ["kmeans (init)"] * len(topq_frac_km) + ["mfa (trained)"] * len(topq_frac_mfa),
            }
        )
        fig = px.histogram(
            topq_long,
            x="topq_fraction",
            color="partition",
            nbins=60,
            barmode="overlay",
            opacity=0.6,
            title=f"Variance fraction captured by the top q={Q} PCs of each cluster",
            color_discrete_map={"kmeans (init)": COLOR_KMEANS, "mfa (trained)": COLOR_MFA},
            template=PLOT_TEMPLATE,
        )
        fig.update_layout(xaxis_title=f"variance fraction in top {Q} PCs", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "spectral_topq_fraction_hist")
        fig.show()

        pr_long = pd.DataFrame(
            {
                "participation_ratio": np.concatenate([pr_km, pr_mfa]),
                "partition": ["kmeans (init)"] * len(pr_km) + ["mfa (trained)"] * len(pr_mfa),
            }
        )
        fig = px.histogram(
            pr_long,
            x="participation_ratio",
            color="partition",
            nbins=60,
            barmode="overlay",
            opacity=0.6,
            title="Spectral participation ratio per cluster (threshold-free effective dim)",
            color_discrete_map={"kmeans (init)": COLOR_KMEANS, "mfa (trained)": COLOR_MFA},
            template=PLOT_TEMPLATE,
        )
        fig.update_layout(xaxis_title="participation ratio of variance spectrum", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "spectral_pr_hist")
        fig.show()


,partition,median variance frac in top 10 PCs,median spectral participation ratio,median rank to 50% var,median rank to 90% var
0,kmeans (init),0.438265,32.737465,15.0,237.0
1,mfa (trained),0.515427,22.988678,10.0,205.0


## 7. Where Do the Learned Subspaces W_k Point?

**Experiment.** Each MFA component owns a rank-q loading matrix `W_k` whose
span is its local subspace. We orthonormalize each `W_k` (QR) and measure the
squared projection of interesting directions onto that span. The reference
value throughout is the **random-subspace baseline q/D ≈ {q}/{D}**: a random
unit vector has expected squared projection q/D onto a random q-dim subspace.

1. **Displacement alignment**: `‖Q_kᵀ (mu_k − c_k)‖² / ‖mu_k − c_k‖²` — did each
   mean move *within* its own learned subspace, or orthogonally to it?
2. **Rogue-dim alignment**: squared projection of the massive-activation axes
   (Section 5) onto each `span(W_k)` — do the learned subspaces spend capacity
   on the rogue channels?
3. **Empirical-PC alignment** (bounded streaming pass): for a random sample of
   clusters, gather activations assigned to the cluster (argmax-responsibility
   assignments, regardless of the `MFA_ASSIGN` selector), compute the top-q
   empirical PCs, and measure (a) the mean squared cosine between `span(W_k)`
   and the empirical PC span, and (b) the within-cluster variance captured by
   `span(W_k)` versus the optimal rank-q capture. This asks: is the learned
   subspace the *right* q-dimensional summary of its own region?


In [ ]:
if not HAVE_MODEL:
    display(Markdown("**Skipped:** missing MFA model files or k-means centroids."))
else:
    from dalg.models.mfa import load_mfa as _load_mfa

    _model = _load_mfa(MFA_RUN / "mfa_model.pt", map_location="cpu")
    with torch.no_grad():
        W_all = _model.W.detach().float()  # (K, D, q)
    del _model
    gc.collect()

    Q_bases, _ = torch.linalg.qr(W_all, mode="reduced")  # (K, D, q)
    q_dim = Q_bases.shape[2]
    D = Q_bases.shape[1]
    RANDOM_BASELINE = q_dim / D
    print(f"span(W_k): q={q_dim}, D={D}, random-subspace baseline q/D = {RANDOM_BASELINE:.4f}")

    disp_t = torch.from_numpy(mu - c_init).float()
    disp_hat = disp_t / disp_t.norm(dim=1, keepdim=True).clamp_min(1e-12)
    proj_disp = torch.einsum("kdq,kd->kq", Q_bases, disp_hat).pow(2).sum(dim=1).numpy()
    print(f"median squared projection of displacement onto span(W_k): {np.median(proj_disp):.3f}")

    if px is not None:
        fig = px.histogram(
            pd.DataFrame({"proj": proj_disp}),
            x="proj",
            nbins=60,
            title="Squared projection of mean displacement (mu_k − c_k) onto span(W_k)",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
        )
        fig.add_vline(
            x=RANDOM_BASELINE, line_dash="dash", line_color="#0b0b0b",
            annotation_text=f"random q/D ({RANDOM_BASELINE:.4f})", annotation_position="top right",
        )
        fig.update_layout(xaxis_title="squared projection onto span(W_k)", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "w_align_displacement_hist")
        fig.show()

    # rogue-dim axes vs learned subspaces (top dims from Section 5)
    rogue_dims = [int(d) for d in top_dims[:5]]
    rogue_proj = pd.concat(
        [
            pd.DataFrame({"dim": f"dim {d}", "proj": Q_bases[:, d, :].pow(2).sum(dim=1).numpy()})
            for d in rogue_dims
        ],
        ignore_index=True,
    )
    display(rogue_proj.groupby("dim")["proj"].describe(percentiles=[0.5, 0.9])[["50%", "90%", "max"]])

    if px is not None:
        fig = px.box(
            rogue_proj,
            x="dim",
            y="proj",
            title="Squared projection of massive-activation axes onto span(W_k)",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
            points=False,
        )
        fig.add_hline(
            y=RANDOM_BASELINE, line_dash="dash", line_color="#0b0b0b",
            annotation_text=f"random q/D ({RANDOM_BASELINE:.4f})", annotation_position="top right",
        )
        fig.update_layout(xaxis_title="rogue dimension", yaxis_title="squared projection onto span(W_k)")
        save_fig(fig, "w_align_rogue_dims")
        fig.show()


span(W_k): q=10, D=2048, random-subspace baseline q/D = 0.0049
median squared projection of displacement onto span(W_k): 0.376


,50%,90%,max
dim,,,
dim 1741,0.051854,0.128843,0.211677
dim 2019,0.037869,0.087756,0.238470
dim 381,0.050059,0.104921,0.185517
dim 50,0.085215,0.130292,0.221313
dim 674,0.423141,0.623330,0.753348


In [ ]:
AR_ASSIGN_PATH = MFA_RUN / _MFA_ASSIGN_FILES["ar"]
if not (HAVE_MODEL and AR_ASSIGN_PATH.exists()):
    display(Markdown("**Skipped:** requires the MFA model and the argmax-responsibility assignments."))
else:
    from dalg.data.shard_activations import ActivationBatchDataset

    SHARD_DIR = REPO / "dalg-cache/pile_gemma2b_activations"
    DROP_PREFIX = json.loads((SHARD_DIR / "config.json").read_text()).get("drop_prefix", 0)

    N_SCAN_TOKENS = 5_000_000       # how much of the stream to scan (bounded cost)
    N_TARGET_CLUSTERS = 48
    MAX_SAMPLES_PER_CLUSTER = 4000
    MIN_SAMPLES_FOR_PCA = 500
    SAMPLE_SEED = 0

    a_ar = torch.load(AR_ASSIGN_PATH, map_location="cpu")["assignments"].to(torch.long)
    head = a_ar[:N_SCAN_TOKENS]
    counts_head = torch.bincount(head, minlength=K)
    eligible = torch.nonzero(counts_head >= MIN_SAMPLES_FOR_PCA).flatten()
    gen = torch.Generator().manual_seed(SAMPLE_SEED)
    targets = eligible[torch.randperm(len(eligible), generator=gen)[:N_TARGET_CLUSTERS]].tolist()
    print(f"sampling {len(targets)} clusters (of {len(eligible)} eligible) from the first {N_SCAN_TOKENS:,} tokens")

    samples = {k: [] for k in targets}
    ds = ActivationBatchDataset(
        SHARD_DIR, LAYER,
        batch_size=131072, drop_prefix=DROP_PREFIX, dtype=torch.float32,
        shuffle_shards=False, shuffle_within_shard=False,
    )
    offset = 0
    for batch in ds:
        n = batch.shape[0]
        assign_slice = a_ar[offset : offset + n]
        for k in targets:
            need = MAX_SAMPLES_PER_CLUSTER - sum(t.shape[0] for t in samples[k])
            if need <= 0:
                continue
            rows = torch.nonzero(assign_slice == k).flatten()
            if len(rows):
                samples[k].append(batch[rows[:need]])
        offset += n
        if offset >= N_SCAN_TOKENS or all(
            sum(t.shape[0] for t in samples[k]) >= MAX_SAMPLES_PER_CLUSTER for k in targets
        ):
            break
    print(f"scanned {offset:,} tokens")

    rows = []
    for k in targets:
        X = torch.cat(samples[k], dim=0)
        if X.shape[0] < MIN_SAMPLES_FOR_PCA:
            continue
        Xc = X - X.mean(dim=0)
        U = torch.linalg.svd(Xc, full_matrices=False).Vh[:q_dim].T  # (D, q) empirical top-q PCs
        Qk = Q_bases[k]
        overlap = float((Qk.T @ U).pow(2).sum() / q_dim)  # mean sq cosine; 1 = same span
        tot = float(Xc.pow(2).sum())
        cap_W = float((Xc @ Qk).pow(2).sum()) / tot
        cap_PC = float((Xc @ U).pow(2).sum()) / tot
        rows.append({"cluster": k, "n_samples": int(X.shape[0]), "overlap": overlap, "cap_W": cap_W, "cap_PC": cap_PC})
    subspace_df = pd.DataFrame(rows)
    subspace_df["capture_ratio"] = subspace_df["cap_W"] / subspace_df["cap_PC"]
    display(subspace_df.describe(percentiles=[0.1, 0.5, 0.9])[["overlap", "cap_W", "cap_PC", "capture_ratio"]])

    if px is not None:
        fig = px.histogram(
            subspace_df,
            x="overlap",
            nbins=30,
            title="Overlap between span(W_k) and top-q empirical PCs of its own region",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
        )
        fig.add_vline(
            x=RANDOM_BASELINE, line_dash="dash", line_color="#0b0b0b",
            annotation_text=f"random q/D ({RANDOM_BASELINE:.4f})", annotation_position="top right",
        )
        fig.update_layout(xaxis_title="mean squared cosine between subspaces (1 = identical span)", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "w_pc_overlap_hist")
        fig.show()

        lim = float(subspace_df[["cap_W", "cap_PC"]].max().max()) * 1.1
        fig = px.scatter(
            subspace_df,
            x="cap_PC",
            y="cap_W",
            hover_data=["cluster", "n_samples", "overlap"],
            title="Within-cluster variance captured: span(W_k) vs optimal rank-q PCs",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
            opacity=0.7,
        )
        fig.add_shape(type="line", x0=0, y0=0, x1=lim, y1=lim, line=dict(color="#0b0b0b", dash="dash", width=1))
        fig.update_layout(
            xaxis_title="variance fraction captured by top-q empirical PCs (upper bound)",
            yaxis_title="variance fraction captured by span(W_k)",
            xaxis_range=[0, lim], yaxis_range=[0, lim],
        )
        save_fig(fig, "w_variance_capture_scatter")
        fig.show()


sampling 48 clusters (of 782 eligible) from the first 5,000,000 tokens
scanned 5,046,272 tokens


,overlap,cap_W,cap_PC,capture_ratio
count,48.000000,48.000000,48.000000,48.000000
mean,0.516063,0.449328,0.601091,0.740300
std,0.125337,0.128784,0.116806,0.116675
min,0.212119,0.166984,0.208280,0.327298
10%,0.338117,0.273766,0.461983,0.606366
50%,0.527969,0.485178,0.632704,0.759655
90%,0.658638,0.600886,0.697626,0.873166
max,0.761511,0.661969,0.786134,0.901470


 `Q_k ` (D×q) is an orthonormal basis for  `span(W_k) ` (from QR) and  `U_k ` (D×q) is an orthonormal basis of the top-q empirical PCs. <br>
overlap =  `‖Q_kᵀ U_k‖²_F / q ` <br>
So cell i,j is the overlap between the learned subspace of MFA component i and the empirical PC subspace of cluster j. <br>
SVD on overlap gives the principal angles between the two subspaces. <br>
Its singular values are exactly  `cos θ₁, …, cos θ_q `. <br>
 `\hat{‖Q_kᵀ U_k‖²_F} = Σᵢ cos² θᵢ ` <br>
 `overlap = mean of cos² ` over the principal angles <br>

## 8. Compact Conclusion

In [ ]:
rows = []
if HAVE_ASSIGNMENTS:
    rows += [
        {"analysis": "assignments", "metric": "same-id agreement", "value": same_id_agreement},
        {"analysis": "assignments", "metric": "NMI", "value": NMI},
        {"analysis": "assignments", "metric": "fraction of clusters self-matched (Hungarian)", "value": frac_self_matched},
        {"analysis": "assignments", "metric": "median per-cluster Jaccard", "value": float(per_cluster["jaccard"].median())},
    ]
if HAVE_MODEL:
    rows += [
        {"analysis": "centroid movement", "metric": "median relative displacement (vs nearest-init distance)", "value": float(np.median(rel_displacement))},
        {"analysis": "centroid movement", "metric": "median centered cos(mu_trained, c_init)", "value": float(np.median(cos_sim_centered))},
        {"analysis": "centroid movement", "metric": "avg centered cos between different k-means centroids", "value": avg_cos_km_centered},
        {"analysis": "centroid movement", "metric": "fraction of trained means nearest own init centroid", "value": frac_nearest_own_init},
        {"analysis": "massive activations", "metric": "dims with RMS > 10x median (kmeans)", "value": n_massive_km},
        {"analysis": "massive activations", "metric": "dims with RMS > 10x median (mfa)", "value": n_massive_mfa},
    ]
if HAVE_IDS:
    rows += [
        {"analysis": "intrinsic dim", "metric": "mean ID kmeans (init)", "value": float(id_km[valid].mean())},
        {"analysis": "intrinsic dim", "metric": "mean ID mfa (trained)", "value": float(id_mfa[valid].mean())},
        {"analysis": "intrinsic dim", "metric": "median ΔID (trained − init)", "value": float(id_df["delta_id"].median())},
    ]
if HAVE_IDS and "topq_frac_km" in globals():
    rows += [
        {"analysis": "spectral", "metric": f"median variance frac in top {Q} PCs (kmeans regions)", "value": float(np.median(topq_frac_km))},
        {"analysis": "spectral", "metric": f"median variance frac in top {Q} PCs (mfa regions)", "value": float(np.median(topq_frac_mfa))},
    ]
if HAVE_MODEL and "proj_disp" in globals():
    rows.append({"analysis": "subspace", "metric": "median sq. projection of displacement onto span(W) (baseline q/D)", "value": float(np.median(proj_disp))})
if HAVE_MODEL and "subspace_df" in globals():
    rows += [
        {"analysis": "subspace", "metric": "median overlap span(W) vs top-q empirical PCs", "value": float(subspace_df["overlap"].median())},
        {"analysis": "subspace", "metric": "median variance-capture ratio span(W) / optimal rank-q", "value": float(subspace_df["capture_ratio"].median())},
    ]
if HAVE_ASSIGNMENTS and HAVE_IDS:
    rows.append(
        {
            "analysis": "coupling",
            "metric": "Spearman(jaccard, |ΔID|)",
            "value": float(merged["jaccard"].corr(merged["abs_delta_id"], method="spearman")),
        }
    )
if rows:
    display(pd.DataFrame(rows))
else:
    display(Markdown("**Skipped:** no artifacts available."))

,analysis,metric,value
0,assignments,same-id agreement,0.573012
1,assignments,NMI,0.723901
2,assignments,fraction of clusters self-matched (Hungarian),0.992000
3,assignments,median per-cluster Jaccard,0.404690
4,centroid movement,median relative displacement (vs nearest-init ...,0.646062
5,centroid movement,"median centered cos(mu_trained, c_init)",0.923085
6,centroid movement,avg centered cos between different k-means cen...,0.006049
7,centroid movement,fraction of trained means nearest own init cen...,0.916000
8,massive activations,dims with RMS > 10x median (kmeans),34.000000
9,massive activations,dims with RMS > 10x median (mfa),30.000000
